In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_HOME"] = "/opt/nvidia/hpc_sdk/Linux_aarch64/24.11/cuda/12.6"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"
os.environ["WANDB_DISABLED"] = "true"

import httpx, torch
print(f"torch {torch.__version__} | cuda devices {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {free//1024**3}GB free / {total//1024**3}GB")
print(f"vLLM: {httpx.get('http://localhost:8000/health/', timeout=5).json()}")


## Run training in-kernel

Calls `inspect_rl_train()` directly — no shell subprocess. Cell blocks until training finishes (~60–90 min for 150 steps). Watch the `[step N] …` lines scroll.


In [ ]:

from inspect_rl.example.math_agent import train

train(
    resume="<redacted-home>/src/inspect-rl/outputs/2026-04-22-17-07-32",
    output_dir="outputs",
    max_steps=150,
)


## Analyse

Reads filesystem. Re-run to get latest val_eval summary. Note: in a blocking Jupyter kernel you can't run this while training is running — either use a second notebook or check eval logs directly from disk.


In [ ]:
from pathlib import Path
from inspect_ai.log import read_eval_log
from inspect_ai.model import ChatMessageAssistant

def summarise(lp):
    log = read_eval_log(str(lp))
    if log.status != "success":
        return f"status={log.status} n={len(log.samples or [])}"
    samples = log.samples or []
    n = len(samples)
    v = lambda s, name: ((s.scores or {}).get(name).value if (s.scores or {}).get(name) else 0)
    correct = sum(v(s, 'correctness') == 1.0 for s in samples)
    calc = sum(v(s, 'uses_calculator') == 1.0 for s in samples)
    valid = sum(v(s, 'valid_submit') == 1.0 for s in samples)
    fails = sum(v(s, 'tool_call_failures') == 1.0 for s in samples)
    avg_chars = sum(sum(len(m.text or "") for m in s.messages if isinstance(m, ChatMessageAssistant)) for s in samples) / max(n, 1)
    return f"n={n:2d} correct={correct:2d}/{n} calc={calc:2d}/{n} valid={valid:2d}/{n} fails={fails:2d}/{n} chars={avg_chars:5.0f}"

ROOT = Path("<redacted-home>/src/inspect-rl/outputs/math_agent_iter6")
if ROOT.exists():
    run_dir = sorted(ROOT.glob("*"))[-1]
    val_logs = sorted((run_dir / "val_eval_logs").glob("*.eval"))
    per_step = sorted((run_dir / "eval_logs").glob("*.eval"))
    print(f"{run_dir.name} — {len(val_logs)} val, {len(per_step)} rollouts")
    for i, lp in enumerate(val_logs):
        print(f"  val {i:2d}: {summarise(lp)}")
else:
    print("no run dir yet")
